In [3]:
import pandas as pd
import numpy as np

# 1. Load Users
users_cols = ['UserID', 'Gender', 'Age', 'Occupation', 'Zip-code']
users = pd.read_csv('ml-1m/users.dat', sep='::', header=None, names=users_cols, engine='python', encoding='latin-1')

# 2. Load Movies
movies_cols = ['MovieID', 'Title', 'Genres']
movies = pd.read_csv('ml-1m/movies.dat', sep='::', header=None, names=movies_cols, engine='python', encoding='latin-1')

# 3. Load Ratings
ratings_cols = ['UserID', 'MovieID', 'Rating', 'Timestamp']
ratings = pd.read_csv('ml-1m/ratings.dat', sep='::', header=None, names=ratings_cols, engine='python', encoding='latin-1')

# 4. Merge them all together into a master dataframe
data = pd.merge(pd.merge(ratings, users), movies)

# Take a look at the first 5 rows
data.head()

,UserID,MovieID,Rating,Timestamp,Gender,Age,Occupation,Zip-code,Title,Genres
0,1,1193,5,978300760,F,1,10,48067,One Flew Over the Cuckoo's Nest (1975),Drama
1,1,661,3,978302109,F,1,10,48067,James and the Giant Peach (1996),Animation|Children's|Musical
2,1,914,3,978301968,F,1,10,48067,My Fair Lady (1964),Musical|Romance
3,1,3408,4,978300275,F,1,10,48067,Erin Brockovich (2000),Drama
4,1,2355,5,978824291,F,1,10,48067,"Bug's Life, A (1998)",Animation|Children's|Comedy


In [4]:
import re

# Extract the 4-digit year from the Title string using Regular Expressions
data['Year'] = data['Title'].str.extract(r'\((\d{4})\)').astype(int)

# Let's drop the Timestamp and Zip-code for now to keep things clean
data = data.drop(columns=['Timestamp', 'Zip-code'])

In [5]:
# Extract the first genre from the string to act as the primary category
data['PrimaryGenre'] = data['Genres'].apply(lambda x: x.split('|')[0])

In [6]:
# Convert Age to string so we can concatenate it
data['Age'] = data['Age'].astype(str)

# Cross 1: Age + Primary Genre (e.g., "18_Action")
data['Age_Genre'] = data['Age'] + "_" + data['PrimaryGenre']

# Cross 2: Gender + Primary Genre (e.g., "M_Action")
data['Gender_Genre'] = data['Gender'] + "_" + data['PrimaryGenre']

In [7]:
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

# 1. Encode Sparse IDs into dense 0-indexed integers
user_encoder = LabelEncoder()
movie_encoder = LabelEncoder()

data['User_Idx'] = user_encoder.fit_transform(data['UserID'])
data['Movie_Idx'] = movie_encoder.fit_transform(data['MovieID'])

# 2. Normalize the 'Year' column (Neural networks hate large numbers like 1995)
# This will scale all years to be between 0.0 and 1.0
scaler = MinMaxScaler()
data['Year_Scaled'] = scaler.fit_transform(data[['Year']])

# Let's peek at our masterpiece
print(data[['User_Idx', 'Movie_Idx', 'Gender_Genre', 'Age_Genre', 'Year_Scaled', 'Rating']].head())

   User_Idx  Movie_Idx Gender_Genre    Age_Genre  Year_Scaled  Rating
0         0       1104      F_Drama      1_Drama     0.691358       5
1         0        639  F_Animation  1_Animation     0.950617       3
2         0        853    F_Musical    1_Musical     0.555556       3
3         0       3177      F_Drama      1_Drama     1.000000       4
4         0       2162  F_Animation  1_Animation     0.975309       5


In [8]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# 1. Encode the crossed strings into integers
ag_encoder = LabelEncoder()
data['Age_Genre_Idx'] = ag_encoder.fit_transform(data['Age_Genre'])

gg_encoder = LabelEncoder()
data['Gender_Genre_Idx'] = gg_encoder.fit_transform(data['Gender_Genre'])

# 2. Re-split the data so the new columns are included in train and test sets
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

# Get the total number of unique categories to define our neural network layer sizes
num_users = len(user_encoder.classes_)
num_movies = len(movie_encoder.classes_)
num_ag = len(ag_encoder.classes_)
num_gg = len(gg_encoder.classes_)

print(f"Vocab sizes -> Users: {num_users}, Movies: {num_movies}, Age_Genre: {num_ag}, Gender_Genre: {num_gg}")

Vocab sizes -> Users: 6040, Movies: 3706, Age_Genre: 126, Gender_Genre: 36


In [9]:
print(f"Training rows: {len(train_data)}")
print(f"Testing rows: {len(test_data)}")

Training rows: 800167
Testing rows: 200042


In [10]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Flatten, Dense, Concatenate
from tensorflow.keras.models import Model

# ==========================================
# 1. DEFINE THE INPUTS
# ==========================================
user_input = Input(shape=(1,), name='User_Idx')
movie_input = Input(shape=(1,), name='Movie_Idx')
year_input = Input(shape=(1,), name='Year_Scaled')
age_genre_input = Input(shape=(1,), name='Age_Genre_Idx')
gender_genre_input = Input(shape=(1,), name='Gender_Genre_Idx')

# ==========================================
# 2. THE WIDE PATH (FTRL Target)
# ==========================================
# Notice all layers here start with "Wide_"
wide_ag = Embedding(input_dim=num_ag, output_dim=1, name='Wide_Age_Genre')(age_genre_input)
wide_gg = Embedding(input_dim=num_gg, output_dim=1, name='Wide_Gender_Genre')(gender_genre_input)

wide_path = Concatenate(name='Wide_Concat')([Flatten()(wide_ag), Flatten()(wide_gg)])

# ==========================================
# 3. THE DEEP PATH (AdaGrad Target)
# ==========================================
# Notice all layers here start with "Deep_"
user_emb = Embedding(input_dim=num_users, output_dim=32, name='Deep_User_Emb')(user_input)
movie_emb = Embedding(input_dim=num_movies, output_dim=32, name='Deep_Movie_Emb')(movie_input)

deep_path = Concatenate(name='Deep_Concat')([Flatten()(user_emb), Flatten()(movie_emb), year_input])
deep_path = Dense(64, activation='relu', name='Deep_Dense_1')(deep_path)
deep_path = Dense(32, activation='relu', name='Deep_Dense_2')(deep_path)

# ==========================================
# 4. FUSION
# ==========================================
combined = Concatenate(name='Fusion_Concat')([wide_path, deep_path])

# We group the final output node with the Deep optimizer
output = Dense(1, activation='linear', name='Deep_Rating_Prediction')(combined)

model = Model(inputs=[user_input, movie_input, year_input, age_genre_input, gender_genre_input], outputs=output)

# We do NOT compile the model here, because we are writing a custom training loop!
model.summary()

Matplotlib is building the font cache; this may take a moment.


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ User_Idx            │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Movie_Idx           │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Deep_User_Emb       │ (None, 1, 32)     │    193,280 │ User_Idx[0][0]    │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Deep_Movie_Emb      │ (None, 1, 32)     │    118,592 │ Movie_Idx[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Age_Genre_Idx       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Gender_Genre_Idx    │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_2 (Flatten) │ (None, 32)        │          0 │ Deep_User_Emb[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_3 (Flatten) │ (None, 32)        │          0 │ Deep_Movie_Emb[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Year_Scaled         │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Wide_Age_Genre      │ (None, 1, 1)      │        126 │ Age_Genre_Idx[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Wide_Gender_Genre   │ (None, 1, 1)      │         36 │ Gender_Genre_Idx… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Deep_Concat         │ (None, 65)        │          0 │ flatten_2[0][0],  │
│ (Concatenate)       │                   │            │ flatten_3[0][0],  │
│                     │                   │            │ Year_Scaled[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 1)         │          0 │ Wide_Age_Genre[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 1)         │          0 │ Wide_Gender_Genr… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Deep_Dense_1        │ (None, 64)        │      4,224 │ Deep_Concat[0][0] │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Wide_Concat         │ (None, 2)         │          0 │ flatten[0][0],    │
│ (Concatenate)       │                   │            │ flatten_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Deep_Dense_2        │ (None, 32)        │      2,080 │ Deep_Dense_1[0][… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Fusion_Concat       │ (None, 34)        │          0 │ Wide_Concat[0][0

 Total params: 318,373 (1.21 MB)

 Trainable params: 318,373 (1.21 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
import time
import tensorflow as tf

# 1. Package the training data into a dictionary
train_inputs = {
    'User_Idx': train_data['User_Idx'].values,
    'Movie_Idx': train_data['Movie_Idx'].values,
    'Year_Scaled': train_data['Year_Scaled'].values,
    'Age_Genre_Idx': train_data['Age_Genre_Idx'].values,
    'Gender_Genre_Idx': train_data['Gender_Genre_Idx'].values
}
train_labels = train_data['Rating'].values

# 2. Package the testing data into a dictionary
test_inputs = {
    'User_Idx': test_data['User_Idx'].values,
    'Movie_Idx': test_data['Movie_Idx'].values,
    'Year_Scaled': test_data['Year_Scaled'].values,
    'Age_Genre_Idx': test_data['Age_Genre_Idx'].values,
    'Gender_Genre_Idx': test_data['Gender_Genre_Idx'].values
}
test_labels = test_data['Rating'].values

# 3. Convert our training dictionary into a high-performance tf.data.Dataset
batch_size = 256
train_dataset = tf.data.Dataset.from_tensor_slices((train_inputs, train_labels))
train_dataset = train_dataset.shuffle(buffer_size=10000).batch(batch_size)

# 4. Define the exact optimizers used in the Google Paper
# FTRL gets L1 regularization to ensure sparsity in the memorized rules
optimizer_wide = tf.keras.optimizers.Ftrl(learning_rate=0.01, l1_regularization_strength=0.001)
optimizer_deep = tf.keras.optimizers.Adagrad(learning_rate=0.01)

# We are predicting a continuous rating (1-5), so we use Mean Squared Error
loss_fn = tf.keras.losses.MeanSquaredError()

# 5. Create a custom training step function
@tf.function
def train_step(inputs, labels):
    with tf.GradientTape() as tape:
        # Forward pass: make predictions
        predictions = model(inputs, training=True)
        # Calculate how far off the predictions were
        loss = loss_fn(labels, predictions)

    # Calculate gradients for the entire model
    gradients = tape.gradient(loss, model.trainable_variables)

    # Separate the variables and their corresponding gradients based on layer names
    wide_vars = []
    wide_grads = []
    deep_vars = []
    deep_grads = []

    for grad, var in zip(gradients, model.trainable_variables):
        # Keras 3 Fix: Check 'path' if it exists, otherwise fallback to 'name'
        name_to_check = getattr(var, 'path', var.name)
        
        if 'Wide_' in name_to_check:
            wide_vars.append(var)
            wide_grads.append(grad)
        else:
            # Everything else (Deep_ layers and Fusion) goes to the Deep optimizer
            deep_vars.append(var)
            deep_grads.append(grad)

    # Apply the FTRL optimizer strictly to the Wide layers
    if wide_grads:
        optimizer_wide.apply_gradients(zip(wide_grads, wide_vars))
        
    # Apply the AdaGrad optimizer strictly to the Deep layers
    if deep_grads:
        optimizer_deep.apply_gradients(zip(deep_grads, deep_vars))

    return loss

# 6. Execute the Training Loop
epochs = 5

print("Starting custom training loop with FTRL (Wide) and AdaGrad (Deep)...")
for epoch in range(epochs):
    start_time = time.time()
    epoch_loss_avg = tf.keras.metrics.Mean()

    # Iterate over the batches of the dataset
    for step, (x_batch_train, y_batch_train) in enumerate(train_dataset):
        loss_value = train_step(x_batch_train, y_batch_train)
        epoch_loss_avg.update_state(loss_value)
        
        # Print progress every 500 batches
        if step % 500 == 0:
            print(f"Epoch {epoch + 1} - Step {step} - Batch Loss: {float(loss_value):.4f}")

    print(f"Epoch {epoch + 1} completed in {time.time() - start_time:.2f}s - Average Loss (MSE): {float(epoch_loss_avg.result()):.4f}")

Starting custom training loop with FTRL (Wide) and AdaGrad (Deep)...
Epoch 1 - Step 0 - Batch Loss: 14.2541
Epoch 1 - Step 500 - Batch Loss: 1.2457
Epoch 1 - Step 1000 - Batch Loss: 1.0074
Epoch 1 - Step 1500 - Batch Loss: 0.9605
Epoch 1 - Step 2000 - Batch Loss: 1.0149
Epoch 1 - Step 2500 - Batch Loss: 0.9599
Epoch 1 - Step 3000 - Batch Loss: 0.7677
Epoch 1 completed in 4.59s - Average Loss (MSE): 1.0879
Epoch 2 - Step 0 - Batch Loss: 0.9010
Epoch 2 - Step 500 - Batch Loss: 0.8243
Epoch 2 - Step 1000 - Batch Loss: 0.6812
Epoch 2 - Step 1500 - Batch Loss: 0.8490
Epoch 2 - Step 2000 - Batch Loss: 0.8090
Epoch 2 - Step 2500 - Batch Loss: 0.8720
Epoch 2 - Step 3000 - Batch Loss: 0.9271
Epoch 2 completed in 4.38s - Average Loss (MSE): 0.8370
Epoch 3 - Step 0 - Batch Loss: 0.9053
Epoch 3 - Step 500 - Batch Loss: 0.7205
Epoch 3 - Step 1000 - Batch Loss: 0.8876
Epoch 3 - Step 1500 - Batch Loss: 0.8787
Epoch 3 - Step 2000 - Batch Loss: 0.7772
Epoch 3 - Step 2500 - Batch Loss: 0.9050
Epoch 3 - 

In [15]:
import joblib # Used for saving our scikit-learn encoders

# 1. Evaluate on the Test Set
print("Evaluating on test data...")
# We use the same loss function (Mean Squared Error) defined earlier
test_predictions = model.predict(test_inputs, batch_size=256)
test_mse = loss_fn(test_labels, test_predictions)
print(f"Final Test MSE: {float(test_mse):.4f}")

# 2. Save the trained TensorFlow model
model.save("wide_and_deep_model.keras")
print("Model saved to 'wide_and_deep_model.keras'")

# 3. Save the Encoders so our API can use them later
joblib.dump(user_encoder, "user_encoder.pkl")
joblib.dump(movie_encoder, "movie_encoder.pkl")
joblib.dump(ag_encoder, "ag_encoder.pkl")
joblib.dump(gg_encoder, "gg_encoder.pkl")
print("Encoders saved successfully!")

Evaluating on test data...
782/782 ━━━━━━━━━━━━━━━━━━━━ 0s 398us/step
Final Test MSE: 0.8279
Model saved to 'wide_and_deep_model.keras'
Encoders saved successfully!


In [16]:
# Grab the first 10 rows from our test data dictionaries
sample_size = 10

sample_inputs = {
    'User_Idx': test_inputs['User_Idx'][:sample_size],
    'Movie_Idx': test_inputs['Movie_Idx'][:sample_size],
    'Year_Scaled': test_inputs['Year_Scaled'][:sample_size],
    'Age_Genre_Idx': test_inputs['Age_Genre_Idx'][:sample_size],
    'Gender_Genre_Idx': test_inputs['Gender_Genre_Idx'][:sample_size]
}

# Grab the actual true ratings for those 10 rows
actual_ratings = test_labels[:sample_size]

# Ask the model to predict the ratings
print("Running predictions...\n")
predicted_ratings = model.predict(sample_inputs, verbose=0)

# Print the comparison neatly
print("-" * 60)
print(f"{'Row':<5} | {'Actual Rating':<15} | {'Predicted Rating':<15}")
print("-" * 60)

for i in range(sample_size):
    # The model outputs a 2D array, so we grab the [0] index to get the raw number
    pred = predicted_ratings[i][0]
    actual = actual_ratings[i]
    
    # We round the prediction to 2 decimal places for readability
    print(f"{i+1:<5} | {actual:<15} | {pred:.2f}")
    
print("-" * 60)

Running predictions...

------------------------------------------------------------
Row   | Actual Rating   | Predicted Rating
------------------------------------------------------------
1     | 2               | 3.60
2     | 5               | 4.90
3     | 4               | 3.43
4     | 4               | 3.39
5     | 1               | 2.73
6     | 4               | 3.76
7     | 5               | 2.29
8     | 3               | 2.85
9     | 1               | 3.12
10    | 1               | 2.68
------------------------------------------------------------
